### Bibliotecas necessárias.

In [ ]:
import pandas as pd
import re
import gc

### Tratamento e limpeza dos dados.

In [ ]:
class OABCleaner:
    """Classe responsável pela sanitização e formatação de strings e dados."""

    LOWERCASE_EXCEPTIONS = {'e', 'o', 'a', 'de', 'do', 'dos', 'da', 'das'}
    CORRECTIONS = {'tributario': 'tributário'}

    @staticmethod
    def clean_category_name(category: str) -> str:
        # Remove números iniciais e sublinhados
        name = re.sub(r'^\d+_?', '', category).replace('_', ' ')

        # Aplica correções específicas e formatação de título
        words = name.split()
        processed = [
            word.lower() if word.lower() in OABCleaner.LOWERCASE_EXCEPTIONS
            else OABCleaner.CORRECTIONS.get(word.lower(), word).title()
            for word in words
        ]
        return ' '.join(processed)

### Classificar questões.

In [ ]:
class OABClassifier:
    """Classe responsável por aplicar métricas de dificuldade e metadados."""

    @staticmethod
    def count_laws(text: str) -> int:
        if not text: return 0
        return len(re.findall(r'\b(lei|leis)\b', str(text).lower()))

    @classmethod
    def apply_difficulty(cls, df: pd.DataFrame) -> pd.DataFrame:
        def get_level(text):
            n = cls.count_laws(text)
            if n == 0: return 1 # Estagiário
            if n <= 2: return 2 # Analista
            if n <= 4: return 3 # Juiz
            return 4            # Ministro

        df['level'] = df['choices'].apply(get_level)
        return df

### Carregar questões e linha guia.

In [ ]:
class OABDataLoader:
    """Classe responsável pelo carregamento e processamento dos datasets."""

    BASE_URL = "https://raw.githubusercontent.com/maritaca-ai/oab-bench/main/data/oab_bench"
    EDITIONS_MAP = {
        '37': 2023, '38': 2023, '39': 2023,
        '40': 2024, '41': 2024, '42': 2024,
        '43': 2025, '44': 2025, '45': 2025
    }

    def __init__(self):
        self.cleaner = OABCleaner()
        self.classifier = OABClassifier()

    def _fetch_jsonl(self, url: str) -> pd.DataFrame:
        return pd.read_json(url, lines=True)

    def load_questions(self) -> pd.DataFrame:
        url = f"{self.BASE_URL}/question.jsonl"
        df = self._fetch_jsonl(url)

        # Adicionar peça prática.
        '''
        brief = ''
        df['brief'] = brief
        for index, row in df.iterrows():
            if 'peca_profissional' in str(row['question_id']):
                brief = row['statement']
            else:
                # Atribui o último 'brief' guardado à linha atual
                df.loc[index, 'brief'] = brief

        # Limpeza e Metadados
        df = df[~df['question_id'].str.contains('peca_profissional')].copy()
        '''
        df['category'] = df['category'].apply(self.cleaner.clean_category_name)
        df['edition'] = df['question_id'].apply(lambda x: x[:2])
        df['year'] = df['edition'].map(self.EDITIONS_MAP).fillna('Desconhecido')
        df['turns'] = df['turns'].apply(lambda x: x[0] if isinstance(x, list) and x else '')

        return df

    def load_guidelines(self) -> pd.DataFrame:
        url = f"{self.BASE_URL}/reference_answer/guidelines.jsonl"
        df = self._fetch_jsonl(url)
        '''
        df = df[~df['question_id'].str.contains('peca_profissional')].copy()
        df['choices'] = df['choices'].apply(
            lambda x: x[0]['turns'][0] if isinstance(x, list) and 'turns' in x[0] else None
        )'''
        return df

    def get_combined_dataset(self, apply_levels=True) -> pd.DataFrame:
        df_q = self.load_questions()
        df_g = self.load_guidelines()

        # Merge
        merged = pd.merge(df_q, df_g[['question_id', 'choices']], on='question_id', how='inner')

        # Seleção de colunas conforme original
        #columns = ['question_id', 'edition', 'year', 'brief', 'statement', 'turns', 'category', 'system', 'choices']
        columns = ['question_id', 'edition', 'year', 'statement', 'turns', 'category', 'system', 'choices']
        merged = merged[columns]

        if apply_levels:
            merged = self.classifier.apply_difficulty(merged)

        merged.insert(0, 'num', range(1, len(merged) + 1))
        merged['type'] = 'Aberta'
        return merged

    @staticmethod
    def save_to_jsonl(df: pd.DataFrame, filename: str):
        try:
            df.to_json(filename, orient='records', lines=True, force_ascii=False)
            print(f"Sucesso: {filename} salvo.")
        except Exception as e:
            print(f"Erro ao salvar: {e}")

### Execução

In [ ]:
loader = OABDataLoader()
df_questions = loader.get_combined_dataset()

# Exemplo: Pegando o subconjunto de 31 a 40
df_my_questions = df_questions.iloc[30:40].copy()

# Exportar
loader.save_to_jsonl(df_questions, 'open_questions.jsonl')

,num,question_id,edition,year,statement,turns,category,system,choices,level,type
0,1,39_direito_administrativo_peca_profissional,39,2023,PEÇA PRÁTICO-PROFISSIONAL\n\nEm fevereiro de 2...,,Direito Administrativo,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['A peça a ser apresent...",4,Aberta
1,2,39_direito_administrativo_questao_1,39,2023,QUESTÃO\n\nA União fez publicar um edital de l...,O edital em questão deveria contemplar a matri...,Direito Administrativo,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['Sim. Nos contratos de...",4,Aberta
2,3,39_direito_administrativo_questao_2,39,2023,QUESTÃO\n\nA sociedade empresária Alfa foi con...,A contratada tem direito à extinção do contrat...,Direito Administrativo,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['Sim, a sociedade empr...",3,Aberta
3,4,39_direito_administrativo_questao_3,39,2023,QUESTÃO\n\nJaqueline é servidora pública ocupa...,"Jaqueline, como agente público responsável pel...",Direito Administrativo,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['Sim, Jaqueline, como ...",2,Aberta
4,5,39_direito_administrativo_questao_4,39,2023,QUESTÃO\n\nApós realizar pedido administrativo...,Qual o prazo para a interposição do recurso ad...,Direito Administrativo,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['Considerando que não ...",4,Aberta
...,...,...,...,...,...,...,...,...,...,...,...
205,206,44_direito_penal_peca_profissional,44,2025,"No dia 10/1/2024, Aluísio, entregador, foi rea...",,Direito Penal,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['De acordo com as info...",3,Aberta
206,207,44_direito_penal_questao_1,44,2025,"Carlos, com a intenção de obter vantagem indev...",A) Qual a tese de Direito Penal deve ser suste...,Direito Penal,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['A questão exige do ex...",1,Aberta
207,208,44_direito_penal_questao_2,44,2025,"Manoela, um dia antes de completar 18 anos, en...","A) A fim de afastar, completamente, a responsa...",Direito Penal,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['A questão exige do ex...",1,Aberta
208,209,44_direito_penal_questao_3,44,2025,"Karina e Daniel, casados, celebraram um contra...",A) Qual a tese de Direito Processual cabível p...,Direito Penal,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['A questão exige do ex...",1,Aberta
